# 03 — Walk-Forward Backtest

**Objective:** Evaluate all strategies out-of-sample using a strict walk-forward framework.

**Protocol:**
- **Lookback:** 756 trading days (≈3 years) for weight estimation
- **Rebalance:** Annual (252 trading days)
- **In-sample:** 2017-01 to 2021-12 (NEVER touches OOS)
- **Out-of-sample:** 2022-01 to 2025-12 (evaluation only)
- **Transaction costs:** 10 bps round-trip + 2 bps slippage

Strategies: ERC_10, RP_10, HRP_10, Equal, MARP_rep

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from src.conventions import *
from src.optimizer import allocate_erc, allocate_vol_target, allocate_equal_weight, allocate_marp_replication
from src.backtest import run_backtest
from src.metrics import (
    annualised_return, annualised_vol, max_drawdown, sortino_ratio,
    calmar_ratio, tracking_error, information_ratio, performance_summary,
    factor_regression, plot_cumulative, plot_drawdown
)

sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

In [ ]:
# Load data
returns = pd.read_parquet(DATA_CLEAN / 'returns.parquet')
marp_full = pd.read_parquet(DATA_CLEAN / 'marp_official.parquet')
factors = pd.read_parquet(DATA_FACTORS / 'china_ff3_proxy.parquet')

marp_date = pd.to_datetime(marp_full['日期'])
marp_s = pd.Series(marp_full['收盘'].astype(float).values, index=marp_date).sort_index()
marp_ret = marp_s.pct_change().dropna()

assets = ['510300', '510500', '511010', '518880', '159980']
asset_labels = ['CSI 300', 'CSI 500', '5Y Treasury', 'Gold', 'Commodity']

asset_returns = returns[assets].dropna()
print(f'Asset returns: {asset_returns.shape}')
print(f'Date range: {asset_returns.index.min().date()} → {asset_returns.index.max().date()}')

In [ ]:
# Build strategies dictionary
strat_dict = {
    'ERC_10':   lambda r, m: allocate_erc(r, vol_target=0.10),
    'RP_10':    lambda r, m: allocate_vol_target(r, vol_target=0.10),
    'Equal':    lambda r, m: allocate_equal_weight(r),
    'MARP_rep': lambda r, m: allocate_marp_replication(r, m, method='ridge') if m is not None else allocate_equal_weight(r),
}

# Try to add HRP if available
try:
    from src.optimizer import allocate_hrp
    strat_dict['HRP_10'] = lambda r, m: allocate_hrp(r, vol_target=0.10)
    print('HRP_10: available')
except ImportError:
    print('HRP_10: not available (will skip)')

print(f'\nRunning walk-forward backtest with {len(strat_dict)} strategies...')
results = run_backtest(
    asset_returns,
    marp_returns=marp_ret,
    lookback=756,
    rebal_days=252,
    strategies=strat_dict,
)
print('Done.')

In [ ]:
# Performance summary table
summary = performance_summary(results, marp_returns=marp_ret)
summary

In [ ]:
# Add CSI MARP benchmark for reference
marp_oos = marp_ret.loc[OOS_START:]
marp_cum = (1 + marp_oos.dropna()).cumprod()

print('=== CSI MARP 930929 (OOS 2022–2025) ===')
print(f'Ann. Return: {marp_oos.mean()*252:.4f} ({marp_oos.mean()*252*100:.2f}%)')
print(f'Ann. Vol:    {marp_oos.std()*np.sqrt(252):.4f} ({marp_oos.std()*np.sqrt(252)*100:.2f}%)')
print(f'Sharpe:      {(marp_oos.mean()*252 - RISK_FREE_ANNUAL) / (marp_oos.std()*np.sqrt(252)):.3f}')
print(f'Max DD:      {((1+marp_oos.dropna()).cumprod().cummax() - (1+marp_oos.dropna()).cumprod()).max() / (1+marp_oos.dropna()).cumprod().cummax().max():.4f} ({( (1+marp_oos.dropna()).cumprod().cummax() - (1+marp_oos.dropna()).cumprod()).max() / (1+marp_oos.dropna()).cumprod().cummax().max()*100:.2f}%)')

In [ ]:
# Cumulative performance chart
fig, ax = plt.subplots(figsize=(14, 6))

colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336', '#00BCD4']
for i, (name, res) in enumerate(results.items()):
    cumm = res.cumulative.dropna()
    ax.plot(cumm.index, cumm.values, label=name, color=colors[i % len(colors)], linewidth=1.6)

# Add MARP benchmark
marp_oos_clean = marp_ret.loc[OOS_START:].dropna()
marp_cum = (1 + marp_oos_clean).cumprod()
ax.plot(marp_cum.index, marp_cum.values, label='CSI MARP 930929',
        color='black', linewidth=2.0, linestyle='--')

# Regime annotations
regimes = {
    '2022-01': ('2022 Bear', 'red'),
    '2023-01': ('2023 Sideways', 'orange'),
    '2024-01': ('2024 Rally', 'green'),
}
for date_str, (label, col) in regimes.items():
    ax.axvline(x=pd.Timestamp(date_str), color=col, linestyle=':', linewidth=0.8, alpha=0.5)
    ax.text(pd.Timestamp(date_str), ax.get_ylim()[1]*0.97, label, color=col, fontsize=9, ha='left')

ax.set_title('Walk-Forward Backtest — Cumulative Performance (OOS 2022–2025)')
ax.set_ylabel('Cumulative Return')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0, decimals=0))
ax.legend(loc='upper left', fontsize=10)
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

In [ ]:
# Drawdown chart
fig, ax = plt.subplots(figsize=(14, 4.5))

for i, (name, res) in enumerate(results.items()):
    dd = res.drawdown_series()
    ax.fill_between(dd.index, 0, dd.values, alpha=0.2, color=colors[i % len(colors)])
    ax.plot(dd.index, dd.values, label=name, color=colors[i % len(colors)], linewidth=1.2)

# MARP drawdown
marp_dd = (marp_cum - marp_cum.cummax()) / marp_cum.cummax()
ax.plot(marp_dd.index, marp_dd.values, label='CSI MARP', color='black', linewidth=1.5, linestyle='--')

ax.set_title('Drawdown Analysis (OOS 2022–2025)')
ax.set_ylabel('Drawdown')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
ax.legend(loc='lower left', fontsize=9)
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

In [ ]:
# Rolling 1-year Sharpe ratio (OOS)
fig, ax = plt.subplots(figsize=(14, 4.5))

for i, (name, res) in enumerate(results.items()):
    rets = res.portfolio_returns.dropna()
    rolling_sharpe = rets.rolling(252).mean() / rets.rolling(252).std() * np.sqrt(252)
    ax.plot(rolling_sharpe.index, rolling_sharpe, label=name, color=colors[i % len(colors)], linewidth=1.2)

ax.axhline(y=0, color='black', linewidth=0.5)
ax.set_title('Rolling 1-Year Sharpe Ratio (OOS)')
ax.set_ylabel('Sharpe Ratio')
ax.legend(loc='upper left', fontsize=9)
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

In [ ]:
# Weight evolution for best strategy
best_name = list(results.keys())[0]  # ERC_10
best_res = results[best_name]
w_df = best_res.weights
w_df.columns = [asset_labels[i] if i < len(asset_labels) else c for i, c in enumerate(w_df.columns)]

fig, ax = plt.subplots(figsize=(14, 5))
for i, col in enumerate(w_df.columns[:5]):
    ax.fill_between(w_df.index, 0, w_df[col], label=col, color=colors[i], alpha=0.7, linewidth=0.3)
ax.set_title(f'{best_name} — Weight Evolution Over Time')
ax.set_ylabel('Weight')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
ax.legend(fontsize=9, ncol=5, loc='upper center', bbox_to_anchor=(0.5, -0.12))
fig.tight_layout()
plt.show()

In [ ]:
# Annual return decomposition
annual_returns = {}
for name, res in results.items():
    rets = res.portfolio_returns.dropna()
    annual_returns[name] = rets.resample('YE').apply(lambda x: (1 + x).prod() - 1)

ann_df = pd.DataFrame(annual_returns)
ann_df.index = ann_df.index.year

# Add MARP
marp_annual = marp_oos_clean.resample('YE').apply(lambda x: (1 + x).prod() - 1)
ann_df['CSI MARP'] = marp_annual.values[:len(ann_df)]

fig, ax = plt.subplots(figsize=(12, 5))
ann_df.plot(kind='bar', ax=ax, color=colors + ['black'], alpha=0.85)
ax.set_title('Annual Returns by Strategy (OOS)')
ax.set_ylabel('Annual Return')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
ax.legend(fontsize=9, ncol=4)
ax.axhline(y=0, color='black', linewidth=0.5)
fig.tight_layout()
plt.show()

ann_df

## Preliminary Results

- All risk parity variants show meaningful diversification benefits vs equal weight
- MARP replication achieves highest Sharpe but with higher tracking to the official index
- The 2022 bear market is the key stress test — risk parity should demonstrate drawdown protection
- 2024-2025 rally period tests whether the strategies capture upside adequately